# Phase 1: Exploratory Data Analysis (EDA)
## Active-Contour-Guided Feature Extraction Pipeline for Automated Polyp Classification

### Objective:
This notebook performs exploratory data analysis on the **Kvasir-SEG** endoscopy dataset. We analyze:
1. **Dataset Size & Integrity**: Image-mask pairing alignment.
2. **Spatial Resolution Heterogeneity**: Image dimensions, aspect ratios, and shape distributions.
3. **Polyp Mask Coverage Ratio**: Relative proportion of endoscopic frame occupied by polyp ground truth.
4. **Sample Visualizations**: RGB frame, binary segmentation mask, and overlay visualizations.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

# Ensure project root is in python path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.config import get_config
from src.data.kvasir_dataset import KvasirSEGDataset

# Configure inline matplotlib plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

--- 
## 1. Dataset Loading & Pair Verification

In [ ]:
# Load configuration and dataset
config = get_config()
dataset = KvasirSEGDataset(config=config)

print(f"Total Number of Samples: {len(dataset)}")
sample = dataset[0]
print(f"Sample Filename        : {sample['filename']}")
print(f"RGB Image Array Shape  : {sample['image'].shape}")
print(f"Binary Mask Shape      : {sample['mask'].shape}")
print(f"Mask Pixel Values      : {np.unique(sample['mask'])}")

--- 
## 2. Image Resolution & Aspect Ratio Analysis

In [ ]:
records = []
for i in range(len(dataset)):
    s = dataset.samples[i]
    img_path = s["image_path"]
    with cv2.imread(str(img_path)) as img:
        pass
    # Get dimensions using fast header read
    from src.utils.io import get_image_dimensions, load_mask
    w, h = get_image_dimensions(img_path)
    mask = load_mask(s["mask_path"])
    polyp_area = np.count_nonzero(mask == 255)
    total_area = w * h
    coverage_pct = (polyp_area / total_area) * 100.0
    
    records.append({
        "filename": s["filename"],
        "width": w,
        "height": h,
        "aspect_ratio": round(w / h, 2),
        "polyp_area_px": polyp_area,
        "total_area_px": total_area,
        "coverage_pct": round(coverage_pct, 2)
    })

df = pd.DataFrame(records)
df.head()

In [ ]:
print("=== Image Resolution Statistics ===")
print(df[['width', 'height', 'aspect_ratio', 'coverage_pct']].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['width'], bins=20, alpha=0.7, color='teal', label='Width')
axes[0].hist(df['height'], bins=20, alpha=0.7, color='coral', label='Height')
axes[0].set_title("Distribution of Image Width and Height (Pixels)")
axes[0].set_xlabel("Pixels")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(df['aspect_ratio'], bins=15, color='purple', alpha=0.7)
axes[1].set_title("Distribution of Aspect Ratios (W / H)")
axes[1].set_xlabel("Aspect Ratio")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

--- 
## 3. Polyp Coverage Percentage Analysis

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['coverage_pct'], bins=30, color='crimson', edgecolor='black', alpha=0.7)
plt.title("Distribution of Ground-Truth Polyp Coverage (% of Total Frame Area)")
plt.xlabel("Polyp Area Coverage (%)")
plt.ylabel("Frequency")
plt.axvline(df['coverage_pct'].mean(), color='blue', linestyle='--', label=f"Mean ({df['coverage_pct'].mean():.2f}%)")
plt.axvline(df['coverage_pct'].median(), color='green', linestyle='-', label=f"Median ({df['coverage_pct'].median():.2f}%)")
plt.legend()
plt.show()

--- 
## 4. Visualizing Sample Image-Mask Pairs & Overlays

In [ ]:
num_samples = 4
indices = np.random.choice(len(dataset), num_samples, replace=False)

fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))

for i, idx in enumerate(indices):
    sample = dataset[idx]
    img = sample['image']
    mask = sample['mask']
    
    # Create RGB Overlay
    overlay = img.copy()
    # Highlight polyp mask area in green tint
    overlay[mask == 255, 1] = np.clip(overlay[mask == 255, 1].astype(int) + 100, 0, 255)
    
    # Draw green contour around polyp boundary
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (255, 0, 0), 3)
    
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"RGB Frame: {sample['filename']}")
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(mask, cmap='gray')
    axes[i, 1].set_title(f"Binary Mask ({mask.shape[1]}x{mask.shape[0]})")
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Polyp Ground Truth Overlay")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()